# NOFE Subject-Level Clustering — Structural Subtypes

Cluster all **16 subjects** by their NOFE embeddings to discover whether subjects with
similar white-matter microstructure form natural groups, and whether those groups
correspond to cognitive profiles.

Pipeline:
```
NOFE embeddings (16 × 32/160) → UMAP / t-SNE (2D) → K-Means + Hierarchical
→ Colour scatter by cluster & by each cognitive score
```

In [1]:
import sys, json, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import kruskal
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# UMAP with graceful fallback to t-SNE
try:
    import umap
    USE_UMAP = True
    print('UMAP available')
except ImportError:
    from sklearn.manifold import TSNE
    USE_UMAP = False
    print('UMAP not found — using t-SNE')

project_root = Path('../../').resolve()
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root / 'dMRI' / 'scripts'))

from utilities import KernelNN
from utilities_dMRI import dMRI_Graph_Generator

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

UMAP available
using Device: mps
Device: mps


## 1. Load NOFE Model

In [2]:
model_type   = 'regular'
model_prefix = 'base'

trained_models_path = project_root / 'dMRI' / 'trained_models' / model_type
model_id = sorted([f.stem for f in trained_models_path.glob(f'{model_prefix}_*.pth')])[-1]
print(f'Model: {model_id}')

with open(trained_models_path / 'logs' / 'configs' / f'{model_id}.json') as f:
    config = json.load(f)

model = KernelNN(
    width=config['width'], ker_width=config['ker_width'], depth=config['depth'],
    in_width=config['node_features'], ker_in=config['edge_attr'], out_width=config['out_width'],
)
model.load_state_dict(torch.load(trained_models_path / f'{model_id}.pth', map_location='cpu'))
model.eval().to(device)
EMBED_DIM = config['width']
print(f'Embedding dim: {EMBED_DIM}')

@torch.no_grad()
def extract_nofe(graph):
    x = model.fc1(graph.x.to(device))
    for _ in range(model.depth):
        x = F.relu(model.conv1(x, graph.edge_index.to(device), graph.edge_attr.to(device)))
    return x.cpu().numpy()

REGION_NAMES = ['frontal', 'parietal', 'temporal', 'occipital', 'subcortical']

def region_masks(coords):
    y, z = coords[:, 1], coords[:, 2]
    return {
        'frontal':     y > 0.60,
        'parietal':    (y > 0.35) & (y < 0.85) & (z > 0.62),
        'temporal':    (y > 0.20) & (y < 0.70) & (z < 0.55),
        'occipital':   y < 0.25,
        'subcortical': (y > 0.35) & (y < 0.65) & (z > 0.28) & (z < 0.58),
    }

def region_mean(hidden, coords):
    masks = region_masks(coords)
    return np.concatenate([
        hidden[m].mean(axis=0) if m.sum() > 0 else np.zeros(EMBED_DIM)
        for m in masks.values()
    ])

Model: base_2026-04-29_t11-16-23
Embedding dim: 32


## 2. Extract / Cache Subject Embeddings

Features are cached to avoid re-running the full extraction each session.

In [3]:
CACHE_DIR  = Path('../cache')
CACHE_DIR.mkdir(exist_ok=True)
CACHE_FILE = CACHE_DIR / 'nofe_subject_features.npz'

data_root    = project_root / 'dMRI' / 'data'
DIFF_SUBPATH = 'T1w/Diffusion_7T/'
N_VOXELS     = 6000
N_REPEATS    = 6

def _flush_mps():
    if device.type == 'mps':
        torch.mps.empty_cache()

local_subjects = sorted([d for d in data_root.iterdir() if d.is_dir() and d.name.isdigit()])

if CACHE_FILE.exists():
    cache = np.load(CACHE_FILE)
    X_global    = cache['X_global']
    X_region    = cache['X_region']
    subject_ids = cache['subject_ids'].tolist()
    print(f'Loaded from cache: {X_global.shape}, {X_region.shape}')
else:
    Generator = dMRI_Graph_Generator(str(data_root))
    global_list, region_list, subject_ids = [], [], []

    for subj_dir in local_subjects:
        path = str(subj_dir / DIFF_SUBPATH) + '/'
        print(f'  {subj_dir.name} ...', end=' ', flush=True)
        Generator.load_data(path=path)
        h_list, c_list = [], []
        for _ in range(N_REPEATS):
            g = Generator.sample_graph(
                N=N_VOXELS, k=config['k'], perplexity=config['perplexity'],
                sampling_method='random', gen_feature_mode=config['feature_mode'],
                sigma_method=config['sigma_method'],
            )
            h = extract_nofe(g)
            c = g.normed_coords.cpu().numpy()
            h_list.append(h); c_list.append(c)
            del g; _flush_mps()
        global_list.append(np.stack([h.mean(axis=0) for h in h_list]).mean(axis=0))
        region_list.append(np.stack([region_mean(h, c) for h, c in zip(h_list, c_list)]).mean(axis=0))
        del h_list, c_list
        subject_ids.append(int(subj_dir.name))
        print('done')
        gc.collect(); _flush_mps()

    X_global = np.stack(global_list)
    X_region = np.stack(region_list)
    np.savez(CACHE_FILE, X_global=X_global, X_region=X_region, subject_ids=np.array(subject_ids))
    print(f'Saved cache: {CACHE_FILE}')

print(f'Subjects: {len(subject_ids)}  |  Global: {X_global.shape}  |  Region: {X_region.shape}')

Loaded from cache: (16, 32), (16, 160)
Subjects: 16  |  Global: (16, 32)  |  Region: (16, 160)


## 3. Load Cognitive Scores

In [4]:
csv_path = sorted(data_root.glob('HCP_YA_subjects_*.csv'))[-1]
df = pd.read_csv(csv_path)

TARGETS = {
    'CardSort':  'CardSort_Unadj',
    'Flanker':   'Flanker_Unadj',
    'PMAT24':    'PMAT24_A_CR',
    'ProcSpeed': 'ProcSpeed_Unadj',
    'ListSort':  'ListSort_Unadj',
    'ReadEng':   'ReadEng_Unadj',
    'PicVocab':  'PicVocab_Unadj',
}
TARGET_NAMES = list(TARGETS.keys())

local_df = df[df['Subject'].isin(subject_ids)].set_index('Subject')
Y = np.stack([local_df.loc[subject_ids, col].values.astype(float)
              for col in TARGETS.values()], axis=1)  # (N, 7)

for i, name in enumerate(TARGET_NAMES):
    print(f'{name:12s}: mean={Y[:,i].mean():.1f}  std={Y[:,i].std():.1f}')

CardSort    : mean=114.4  std=11.4
Flanker     : mean=109.3  std=7.5
PMAT24      : mean=14.1  std=5.3
ProcSpeed   : mean=115.5  std=17.9
ListSort    : mean=109.3  std=12.2
ReadEng     : mean=118.2  std=12.1
PicVocab    : mean=113.9  std=10.1


## 4. 2D Dimensionality Reduction (UMAP / t-SNE)

In [ ]:
FEAT_SETS = {
    'Global (32d)': X_global,
    'Region (160d)': X_region,
}

embeddings_2d = {}
for fname, X in FEAT_SETS.items():
    Xsc = StandardScaler().fit_transform(X)
    if USE_UMAP:
        reducer = umap.UMAP(n_components=2, random_state=0, n_neighbors=8, min_dist=0.2)
        emb = reducer.fit_transform(Xsc)
        method = 'UMAP'
    else:
        reducer = TSNE(n_components=2, random_state=0, perplexity=5)
        emb = reducer.fit_transform(Xsc)
        method = 't-SNE'
    embeddings_2d[fname] = emb
    print(f'{fname} → {method} done')

print(f'Reduction method: {method}')

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Global (32d) → UMAP done
Region (160d) → UMAP done
Reduction method: UMAP


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


: 

## 5. K-Means on 2D Embedding + Silhouette

In [ ]:
K_range = range(2, min(7, len(subject_ids)))

fig, axes = plt.subplots(len(FEAT_SETS), len(K_range),
                          figsize=(len(K_range) * 3, len(FEAT_SETS) * 3.2),
                          facecolor='white')

best_k_per_feat = {}
for row, (fname, X) in enumerate(FEAT_SETS.items()):
    emb = embeddings_2d[fname]
    Xsc = StandardScaler().fit_transform(X)
    sils = []
    for col, k in enumerate(K_range):
        km  = KMeans(n_clusters=k, random_state=0, n_init=20)
        lbl = km.fit_predict(Xsc)
        sil = silhouette_score(Xsc, lbl) if k > 1 else 0.0
        sils.append(sil)
        ax = axes[row, col] if len(FEAT_SETS) > 1 else axes[col]
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=lbl, cmap='Set1',
                        s=100, edgecolors='white', linewidths=0.8)
        for i, sid in enumerate(subject_ids):
            ax.annotate(str(sid)[-4:], (emb[i, 0], emb[i, 1]),
                        fontsize=5.5, ha='center', va='bottom', color='#333')
        ax.set_title(f'{fname}\nK={k}  sil={sil:.3f}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    best_k = list(K_range)[np.argmax(sils)]
    best_k_per_feat[fname] = best_k
    print(f'{fname}: best K={best_k}  (silhouette={max(sils):.3f})')

plt.suptitle(f'{method} + K-Means  (label = last 4 digits of subject ID)',
             fontsize=12, y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Hierarchical Clustering + Dendrogram

In [ ]:
fig, axes = plt.subplots(1, len(FEAT_SETS), figsize=(7 * len(FEAT_SETS), 5),
                          facecolor='white')
if len(FEAT_SETS) == 1:
    axes = [axes]

for ax, (fname, X) in zip(axes, FEAT_SETS.items()):
    Xsc = StandardScaler().fit_transform(X)
    Z   = linkage(pdist(Xsc, metric='cosine'), method='ward')
    labels_str = [str(sid)[-4:] for sid in subject_ids]
    dendrogram(Z, labels=labels_str, ax=ax, color_threshold=0.7*max(Z[:,2]),
               leaf_font_size=9)
    ax.set_title(f'Hierarchical clustering — {fname}\n(cosine distance, Ward linkage)',
                 fontsize=11)
    ax.set_ylabel('Distance')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. 2D Scatter Colored by Cognitive Scores

In [ ]:
# Use the feature set with best silhouette for main plot
BEST_FEAT = max(best_k_per_feat, key=lambda f: best_k_per_feat[f])
emb = embeddings_2d[BEST_FEAT]
Xsc = StandardScaler().fit_transform(FEAT_SETS[BEST_FEAT])
km  = KMeans(n_clusters=best_k_per_feat[BEST_FEAT], random_state=0, n_init=20)
clusters = km.fit_predict(Xsc)

ncols = 4
nrows = (len(TARGET_NAMES) + ncols) // ncols  # +1 row for cluster
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 3.8, nrows * 3.5),
                          facecolor='white')
axes = np.array(axes).reshape(-1)

# First panel: cluster assignment
ax = axes[0]
sc = ax.scatter(emb[:, 0], emb[:, 1], c=clusters, cmap='Set1',
                s=120, edgecolors='white', linewidths=1)
for i, sid in enumerate(subject_ids):
    ax.annotate(str(sid)[-4:], (emb[i, 0], emb[i, 1]),
                fontsize=6, ha='center', va='bottom')
ax.set_title(f'K-Means clusters (K={best_k_per_feat[BEST_FEAT]})', fontsize=10, fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(sc, ax=ax, fraction=0.04, label='Cluster')

# Remaining panels: cognitive scores as colour
for i, name in enumerate(TARGET_NAMES):
    ax  = axes[i + 1]
    sc2 = ax.scatter(emb[:, 0], emb[:, 1], c=Y[:, i],
                     cmap='RdYlGn', s=120, edgecolors='white', linewidths=1)
    for j, sid in enumerate(subject_ids):
        ax.annotate(str(sid)[-4:], (emb[j, 0], emb[j, 1]),
                    fontsize=6, ha='center', va='bottom')
    ax.set_title(name, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(sc2, ax=ax, fraction=0.04)

for ax in axes[len(TARGET_NAMES) + 1:]:
    ax.axis('off')

fig.suptitle(f'{method} embedding of {BEST_FEAT}  |  colour = score value',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Cluster Statistics — Mean Scores per Group

In [ ]:
K_FINAL = best_k_per_feat[BEST_FEAT]
cluster_col = f'Cluster (K={K_FINAL})'

df_subj = pd.DataFrame(Y, columns=TARGET_NAMES, index=subject_ids)
df_subj[cluster_col] = clusters
df_subj['SubjectID']  = subject_ids

print('Mean cognitive scores per cluster:')
print(df_subj.groupby(cluster_col)[TARGET_NAMES].mean().round(1))

# Kruskal-Wallis test per score
print('\nKruskal-Wallis p-values (non-parametric, interpret with caution — n=16):')
for name in TARGET_NAMES:
    groups = [df_subj.loc[df_subj[cluster_col] == c, name].values for c in range(K_FINAL)]
    if all(len(g) > 0 for g in groups):
        stat, p = kruskal(*groups)
        sig = ' *' if p < 0.05 else ''
        print(f'  {name:12s}: H={stat:.2f}  p={p:.3f}{sig}')

# Radar / heatmap of cluster means
fig, ax = plt.subplots(figsize=(10, 3.5), facecolor='white')
group_means = df_subj.groupby(cluster_col)[TARGET_NAMES].mean()
# Normalise each score to [0,1] for display
group_means_n = (group_means - group_means.min()) / (group_means.max() - group_means.min() + 1e-9)
im = ax.imshow(group_means_n.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(TARGET_NAMES)))
ax.set_xticklabels(TARGET_NAMES, rotation=30, ha='right')
ax.set_yticks(range(K_FINAL))
ax.set_yticklabels([f'Cluster {c} (n={int((clusters==c).sum())})' for c in range(K_FINAL)])
for i in range(K_FINAL):
    for j in range(len(TARGET_NAMES)):
        v = group_means.values[i, j]
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=9,
                color='white' if group_means_n.values[i,j] < 0.3 else '#222')
plt.colorbar(im, ax=ax, label='Normalised score (within column)', fraction=0.015)
ax.set_title('Cluster mean cognitive scores (colour = relative level within each score)', fontsize=11)
plt.tight_layout()
plt.show()